In [1]:
from pathlib import Path
import os
from neuralhydrology.evaluation import metrics, get_tester
from neuralhydrology.utils.config import Config
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pandas as pd
import yaml
import warnings
warnings.simplefilter("ignore", FutureWarning)
import plotly.graph_objects as go
import geopandas as gpd
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import copy
import xarray as xr
import neuralhydrology

Allereerst bepalen wij bij welke epoch de modellen de hoogste NSE hebben, daarna plotten we de scores

In [2]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
import os 
def extract_tensorboard_scalars(logdir):
    event_acc = EventAccumulator(logdir)
    event_acc.Reload()  # Load the TensorBoard logs

    scalars = {}
    for tag in event_acc.Tags()['scalars']:  # Get all scalar tags
        scalars[tag] = [(e.step, e.value) for e in event_acc.Scalars(tag)]
    
    return scalars

Hieronder base_folder aanpassen naar je eigen mapje.

In [16]:
base_folder = r'C:\Users\christiaan.wewer\Documents\projecten\hdsr_lstm_2026_1\code\neural_hydrology\scripts\analysis\visualisatie_chris_meeting'

In [17]:
runs_folder = r'runs'
run_names = [f for f in os.listdir(os.path.join(base_folder, runs_folder)) if os.path.isdir(os.path.join(base_folder, runs_folder, f))]
run_names

['development_run_33_1004_025034',
 'trial_33_retrain_1_1104_124653',
 'trial_33_retrain_2_1104_130740',
 'trial_33_retrain_3_1104_132843',
 'trial_33_retrain_4_1104_135000',
 'trial_33_retrain_5_1104_141137']

## run volgende cellen alleen 1x als je de resultaten nog niet opgeslagen hebt

In [5]:
# # copy the entire runs folder to a folder called runs_backup
# runs_backup_folder = os.path.join(base_folder, 'runs_backup')
# if not os.path.exists(runs_backup_folder):
#     os.makedirs(runs_backup_folder)
# os.system(f'cp -r {os.path.join(base_folder, runs_folder)} {runs_backup_folder}')

# print(f"Backup created successfully at: {runs_backup_folder}")


In [20]:
# # get NSE curves of validation data
run_info = []

for r in run_names:

    folder = f'{base_folder}\{runs_folder}\{r}'
    data = extract_tensorboard_scalars(folder)

    # break
    try:
        nse_vals_1d = np.array([score for epoch, score in data['valid/mean_nse_1d']]) 
        nse_vals_1h = np.array([score for epoch, score in data['valid/mean_nse_1h']]) 
        nse_vals_avg = (nse_vals_1d + nse_vals_1h) / 2
        max_avg_nse_epoch = (np.argmax(nse_vals_avg)+1) # +1 because of 0 indexing, * 5 because we save per 5 epochs

        run_info.append({
            'Run Name': r,
            'Epoch With Max AVG NSE': max_avg_nse_epoch,
            'Max AVG NSE Value': np.max(nse_vals_avg),
            'Epoch With Max 1H NSE': np.argmax(nse_vals_1h) + 1,
            'Max 1H NSE Value': np.max(nse_vals_1h),
            'Epoch With Max 1D NSE': np.argmax(nse_vals_1d) + 1,
            'Max 1D NSE Value': np.max(nse_vals_1d),
        })

        print(f"Run {r} processed successfully.")

    except KeyError:
        run_names.remove(r)
        print(f"KeyError for run {r}, skipping this run.")

Run development_run_33_1004_025034 processed successfully.
Run trial_33_retrain_1_1104_124653 processed successfully.
Run trial_33_retrain_2_1104_130740 processed successfully.
Run trial_33_retrain_3_1104_132843 processed successfully.
Run trial_33_retrain_4_1104_135000 processed successfully.
Run trial_33_retrain_5_1104_141137 processed successfully.


In [10]:
data.keys()

dict_keys(['train/loss', 'train/total_loss', 'train/avg_loss', 'train/avg_total_loss', 'valid/avg_loss', 'valid/avg_total_loss', 'valid/mean_nse_1d', 'valid/median_nse_1d', 'valid/mean_nse_1h', 'valid/median_nse_1h'])

De NSE values beneden zijn op de validatie set en dus niet het uiteindelijke resultaat

In [21]:
df_runs = pd.DataFrame(run_info)
df_runs.sort_values(by='Max AVG NSE Value', ascending=False)

,Run Name,Epoch With Max AVG NSE,Max AVG NSE Value,Epoch With Max 1H NSE,Max 1H NSE Value,Epoch With Max 1D NSE,Max 1D NSE Value
0,development_run_33_1004_025034,45,0.262657,40,0.289530,45,0.254563
2,trial_33_retrain_2_1104_130740,44,0.219387,19,0.305958,44,0.198798
3,trial_33_retrain_3_1104_132843,34,0.203858,24,0.265465,34,0.175836
1,trial_33_retrain_1_1104_124653,68,0.193750,74,0.267801,68,0.133834
5,trial_33_retrain_5_1104_141137,31,0.189189,15,0.252243,30,0.211798
4,trial_33_retrain_4_1104_135000,71,0.185615,14,0.221196,71,0.164322


Neuralhydrology pakt van zichzelf altijd het model van de laatste epoch beschikbaar, maar dat is wellicht niet het model met de hoogste NSE score op de validatie set.
Daarom checken wij eerst welke epoch de hoogste NSE heeft en kopieren wij scriptmatig alle laatste epochs naar een mapje

In [22]:
# per folder in run_names I want to make a backup folder if it does not exist yet. 
# Then I want to copy all the files that end with epochXXX.pt to the backup folder, 
# where XXX are all numbers that are higher than the corresponding max_nse_per_epoch
for r, max_nse_epoch in zip( df_runs['Run Name'].values, df_runs['Epoch With Max 1H NSE'].values):
    folder = f'{base_folder}\{runs_folder}\{r}'
    backup_folder = f'{base_folder}\{runs_folder}\{r}\\backup'
    if not os.path.exists(backup_folder):
        os.makedirs(backup_folder)
    for file in os.listdir(folder):
        if file.endswith('.pt'):
            epoch = int(file.split('epoch')[-1].split('.')[0])
            if epoch > max_nse_epoch:
                print(file, epoch)

                # CUT file to backup folder
                os.rename(f'{folder}\{file}', f'{backup_folder}\{file}')



model_epoch041.pt 41
model_epoch042.pt 42
model_epoch043.pt 43
model_epoch044.pt 44
model_epoch045.pt 45
model_epoch046.pt 46
model_epoch047.pt 47
model_epoch048.pt 48
model_epoch049.pt 49
model_epoch050.pt 50
model_epoch051.pt 51
model_epoch052.pt 52
model_epoch053.pt 53
model_epoch054.pt 54
model_epoch055.pt 55
model_epoch056.pt 56
model_epoch057.pt 57
model_epoch058.pt 58
model_epoch059.pt 59
model_epoch060.pt 60
model_epoch061.pt 61
model_epoch062.pt 62
model_epoch063.pt 63
model_epoch064.pt 64
model_epoch065.pt 65
model_epoch066.pt 66
model_epoch067.pt 67
model_epoch068.pt 68
model_epoch069.pt 69
model_epoch070.pt 70
model_epoch071.pt 71
model_epoch072.pt 72
model_epoch073.pt 73
model_epoch074.pt 74
model_epoch075.pt 75
optimizer_state_epoch041.pt 41
optimizer_state_epoch042.pt 42
optimizer_state_epoch043.pt 43
optimizer_state_epoch044.pt 44
optimizer_state_epoch045.pt 45
optimizer_state_epoch046.pt 46
optimizer_state_epoch047.pt 47
optimizer_state_epoch048.pt 48
optimizer_state_e

In [23]:
run_names

['development_run_33_1004_025034',
 'trial_33_retrain_1_1104_124653',
 'trial_33_retrain_2_1104_130740',
 'trial_33_retrain_3_1104_132843',
 'trial_33_retrain_4_1104_135000',
 'trial_33_retrain_5_1104_141137']

In [27]:
from pathlib import Path

def adapt_config_for_local_visualisation(config: dict, base_folder: str | Path) -> dict:
    """Rewrite Databricks paths in a run config to the local visualisation folder layout."""
    base_folder = Path(base_folder).expanduser().resolve()

    adapted = dict(config)

    if "run_dir" not in adapted or not adapted["run_dir"]:
        raise KeyError("config['run_dir'] is required to infer the local run folder.")

    run_name = Path(str(adapted["run_dir"])).name
    local_run_dir = base_folder / "runs" / run_name
    local_data_dir = base_folder / "data"
    local_basin_file = base_folder / "hdsr_polders.txt"

    adapted.update({
        "data_dir": local_data_dir,
        "run_dir": local_run_dir,
        "train_dir": local_run_dir / "train_data",
        "img_log_dir": local_run_dir / "img_log",
        "train_basin_file": local_basin_file,
        "validation_basin_file": local_basin_file,
        "test_basin_file": local_basin_file,
    })

    required_paths = [
        local_data_dir,
        local_run_dir,
        local_run_dir / "train_data",
        local_basin_file,
    ]
    missing = [path for path in required_paths if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required local paths:\n" + "\n".join(f"- {path}" for path in missing)
        )

    return adapted


In [29]:
# reload tester
# %load_ext autoreload
# import importlib
# import neuralhydrology.evaluation
# import neuralhydrology.utils.config

# importlib.reload(neuralhydrology.evaluation)
# importlib.reload(neuralhydrology.utils.config)

# get_tester = neuralhydrology.evaluation.get_tester
# Config = neuralhydrology.utils.config.Config


# results_list_path: str = os.path.join(base_folder, 'results_list_final_runs.npy')

# uncomment below to recompute results

run_type = ['validation', 'test']
results_list = []
for run in run_names:
    results_val_test = []
    for rt in run_type:
        run_folder_path = Path(runs_folder + '/' + run)
        with open(run_folder_path / 'config.yml') as file:
            config = yaml.load(file, Loader=yaml.FullLoader)
        # config['epochs'] = 1
        config = adapt_config_for_local_visualisation(config, base_folder)

        config['device'] = 'cpu'

        print(run)
        tester_config = Config(config)
        tester = get_tester(tester_config, run_folder_path, init_model=True, period=rt)
        results = tester.evaluate(save_results=True, metrics=tester_config.metrics)
        results_val_test.append(results)
    results_list.append(results_val_test)

development_run_33_1004_025034
# Validation: 100%|██████████| 40/40 [00:36<00:00,  1.09it/s]
development_run_33_1004_025034
# Evaluation: 100%|██████████| 40/40 [00:44<00:00,  1.10s/it]
trial_33_retrain_1_1104_124653
# Validation: 100%|██████████| 40/40 [00:36<00:00,  1.10it/s]
trial_33_retrain_1_1104_124653
# Evaluation: 100%|██████████| 40/40 [00:33<00:00,  1.20it/s]
trial_33_retrain_2_1104_130740
# Validation: 100%|██████████| 40/40 [00:30<00:00,  1.30it/s]
trial_33_retrain_2_1104_130740
# Evaluation: 100%|██████████| 40/40 [00:33<00:00,  1.18it/s]
trial_33_retrain_3_1104_132843
# Validation: 100%|██████████| 40/40 [00:27<00:00,  1.45it/s]
trial_33_retrain_3_1104_132843
# Evaluation: 100%|██████████| 40/40 [00:32<00:00,  1.24it/s]
trial_33_retrain_4_1104_135000
# Validation: 100%|██████████| 40/40 [00:29<00:00,  1.36it/s]
trial_33_retrain_4_1104_135000
# Evaluation: 100%|██████████| 40/40 [00:27<00:00,  1.47it/s]
trial_33_retrain_5_1104_141137
# Validation: 100%|██████████| 40/40 [0

In [ ]:
results['AFVG26']['1h']

{'xr': <xarray.Dataset> Size: 146kB
 Dimensions:     (date: 731, time_step: 24)
 Coordinates:
   * date        (date) datetime64[ns] 6kB 2021-01-01 2021-01-02 ... 2023-01-01
   * time_step   (time_step) int64 192B 0 1 2 3 4 5 6 7 ... 17 18 19 20 21 22 23
 Data variables:
     afvoer_obs  (date, time_step) float32 70kB 4.664e-08 4.664e-08 ... nan nan
     afvoer_sim  (date, time_step) float32 70kB 3.782e-08 3.724e-08 ... nan nan,
 'NSE_1h': 0.3141358494758606}

First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': None, 'NSE_1h': None, 'NSE_AVG_1D_1h': None}
First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': None, 'NSE_1h': None, 'NSE_AVG_1D_1h': None}
First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': None, 'NSE_1h': None, 'NSE_AVG_1D_1h': None}
First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': None, 'NSE_1h': None, 'NSE_AVG_1D_1h': None}
First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': None, 'NSE_1h': None, 'NSE_AVG_1D_1h': None}
First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': None, 'NSE_1h': None, 'NSE_AVG_1D_1h': None}
First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': None, 'NSE_1h': None, 'NSE_AVG_1D_1h': None}
First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': None, 'NSE_1h': None, 'NSE_AVG_1D_1h': None}
First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': None, 'NSE_1h': None, 'NSE_AVG_1D_1h': None}
F

In [53]:
results_list[0][1]['AFVG1']['1h']

{'xr': <xarray.Dataset> Size: 146kB
 Dimensions:     (date: 731, time_step: 24)
 Coordinates:
   * date        (date) datetime64[ns] 6kB 2021-01-01 2021-01-02 ... 2023-01-01
   * time_step   (time_step) int64 192B 0 1 2 3 4 5 6 7 ... 17 18 19 20 21 22 23
 Data variables:
     afvoer_obs  (date, time_step) float32 70kB nan nan nan nan ... nan nan nan
     afvoer_sim  (date, time_step) float32 70kB nan nan nan nan ... nan nan nan}

# load results

In [ ]:
def build_results_list_with_ensemble(results_list, n_members=None):
    """
    Returns a new results list with two extra pseudo-models appended:
    - second-to-last: mean ensemble
    - last: median ensemble

    The input results_list is not modified.
    """
    import warnings
    import numpy as np

    def _(obs, sim):
        obs_mean = np.nanmean(obs)
        numerator = np.nansum((obs - sim) ** 2)
        denominator = np.nansum((obs - obs_mean) ** 2)
        return 1 - (numerator / denominator)

    def _reduce_without_warnings(stack, reducer):
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            return reducer(stack, axis=0)

    source_results = list(results_list[:n_members]) if n_members is not None else list(results_list)
    if len(source_results) == 0:
        raise ValueError("results_list is empty.")

    def _build_ensemble_member(reducer):
        ensemble_member = []

        for period_idx in range(len(source_results[0])):
            period_result = {}

            for series_key, freq_dict in source_results[0][period_idx].items():
                period_result[series_key] = {}

                for freq_key, template_entry in freq_dict.items():
                    xr_template = template_entry["xr"]

                    sim_stack = np.stack([
                        model_result[period_idx][series_key][freq_key]["xr"]["afvoer_sim"].values
                        for model_result in source_results
                    ], axis=0)

                    ensemble_sim = _reduce_without_warnings(sim_stack, reducer)

                    xr_ensemble = xr_template.copy(deep=True)
                    xr_ensemble["afvoer_sim"] = (
                        xr_template["afvoer_sim"].dims,
                        ensemble_sim.astype(xr_template["afvoer_sim"].dtype, copy=False)
                    )

                    ensemble_entry = {"xr": xr_ensemble}
                    for metric_key in template_entry.keys():
                        if metric_key.startswith("NSE_"):
                            obs = xr_ensemble["afvoer_obs"].values.reshape(-1)
                            sim = xr_ensemble["afvoer_sim"].values.reshape(-1)
                            ensemble_entry[metric_key] = float(_compute_nse(obs, sim))

                    period_result[series_key][freq_key] = ensemble_entry

            ensemble_member.append(period_result)

        return ensemble_member

    results_list_with_ensemble = list(source_results)
    results_list_with_ensemble.append(_build_ensemble_member(np.nanmean))
    results_list_with_ensemble.append(_build_ensemble_member(np.nanmedian))

    return results_list_with_ensemble

results_with_ensemble = build_results_list_with_ensemble(results_list)

In [ ]:
from pathlib import Path
import pandas as pd


def save_period_results_to_netcdf(
    period_results: dict,
    netcdf_output_dir: str | Path,
    time_resolution: str = "1h",
    filename_template: str = "simulation_output_{basin}.nc",
):
    """
    Save one period result dict to one NetCDF per basin.

    Example input:
        period_results = results_with_ensemble[-2][1]
        # avg ensemble, test set

    Parameters
    ----------
    period_results : dict
        Dict like results_list[i][period_idx], keyed by basin.
    netcdf_output_dir : str | Path
        Output folder for the NetCDF files.
    time_resolution : str
        Usually "1h" or "1D".
    filename_template : str
        Output filename pattern. Must contain "{basin}".
    """
    netcdf_output_dir = Path(netcdf_output_dir)
    netcdf_output_dir.mkdir(parents=True, exist_ok=True)

    time_unit_map = {
        "1h": "h",
        "1D": "D",
    }
    if time_resolution not in time_unit_map:
        raise ValueError("time_resolution must be '1h' or '1D'.")

    written_files = []

    for basin, basin_results in period_results.items():
        if time_resolution not in basin_results:
            raise KeyError(f"Missing time resolution {time_resolution!r} for basin {basin}.")

        ds = basin_results[time_resolution]["xr"].copy(deep=True)

        # Match the export shape used elsewhere in the repo:
        # convert (date, time_step) into a single datetime index.
        if "time_step" in ds.dims and time_resolution == "1h" and ds.sizes["time_step"] > 24:
            ds = ds.isel(time_step=slice(-24, None))

        if {"date", "time_step"}.issubset(ds.coords):
            ds = ds.stack(datetime=("date", "time_step")).reset_index("datetime")
            ds["datetime"] = ds["date"] + pd.to_timedelta(
                ds["time_step"].astype(int),
                unit=time_unit_map[time_resolution],
            )
            ds = ds.set_index({"datetime": "datetime"}).drop_vars(["date", "time_step"])

        output_path = netcdf_output_dir / filename_template.format(
            basin=basin,
            time_resolution=time_resolution,
        )
        ds.to_netcdf(output_path)
        written_files.append(output_path)

    return written_files


dict_keys(['AFVG1', 'AFVG13', 'AFVG15', 'AFVG16', 'AFVG18', 'AFVG22', 'AFVG23', 'AFVG24', 'AFVG26', 'AFVG28', 'AFVG29', 'AFVG30', 'AFVG31', 'AFVG33', 'AFVG34', 'AFVG36', 'AFVG37', 'AFVG39', 'AFVG41', 'AFVG42', 'AFVG44', 'AFVG45', 'AFVG46', 'AFVG47', 'AFVG5', 'AFVG50', 'AFVG54', 'AFVG55', 'AFVG56', 'AFVG57', 'AFVG58', 'AFVG6', 'AFVG62', 'AFVG64', 'AFVG65', 'AFVG67', 'AFVG70', 'AFVG77', 'AFVG78', 'AFVG8'])

In [51]:
from pathlib import Path
import pandas as pd


def save_period_results_to_netcdf(
    period_results: dict,
    netcdf_output_dir: str | Path,
    time_resolution: str = "1h",
    filename_template: str = "simulation_output_{basin}.nc",
):
    """
    Save one period result dict to one NetCDF per basin.

    Example input:
        period_results = results_with_ensemble[-2][1]
        # avg ensemble, test set

    Parameters
    ----------
    period_results : dict
        Dict like results_list[i][period_idx], keyed by basin.
    netcdf_output_dir : str | Path
        Output folder for the NetCDF files.
    time_resolution : str
        Usually "1h" or "1D".
    filename_template : str
        Output filename pattern. Must contain "{basin}".
    """
    netcdf_output_dir = Path(netcdf_output_dir)
    netcdf_output_dir.mkdir(parents=True, exist_ok=True)

    time_unit_map = {
        "1h": "h",
        "1D": "D",
    }
    if time_resolution not in time_unit_map:
        raise ValueError("time_resolution must be '1h' or '1D'.")

    written_files = []

    for basin, basin_results in period_results.items():
        if time_resolution not in basin_results:
            raise KeyError(f"Missing time resolution {time_resolution!r} for basin {basin}.")

        ds = basin_results[time_resolution]["xr"].copy(deep=True)

        # Match the export shape used elsewhere in the repo:
        # convert (date, time_step) into a single datetime index.
        if "time_step" in ds.dims and time_resolution == "1h" and ds.sizes["time_step"] > 24:
            ds = ds.isel(time_step=slice(-24, None))

        if {"date", "time_step"}.issubset(ds.coords):
            ds = ds.stack(datetime=("date", "time_step")).reset_index("datetime")
            ds["datetime"] = ds["date"] + pd.to_timedelta(
                ds["time_step"].astype(int),
                unit=time_unit_map[time_resolution],
            )
            ds = ds.set_index({"datetime": "datetime"}).drop_vars(["date", "time_step"])

        output_path = netcdf_output_dir / filename_template.format(
            basin=basin,
            time_resolution=time_resolution,
        )
        ds.to_netcdf(output_path)
        written_files.append(output_path)

    return written_files


In [52]:
netcdf_output_dir = Path(base_folder) / "results_netcdf"
written_files = save_period_results_to_netcdf(
    results_with_ensemble[-2][1],  # avg ensemble, test set
    netcdf_output_dir=netcdf_output_dir,
    time_resolution="1h",
)


In [36]:
def compute_NSE(obs, sim):
    """
    Compute the Nash Sutcliffe Efficiency (NSE) between observed and simulated values.
    """
    obs_mean = np.nanmean(obs)
    numerator = np.nansum((obs - sim) ** 2)
    denominator = np.nansum((obs - obs_mean) ** 2)
    nse = 1 - (numerator / denominator)
    return nse


In [75]:
df_attributes = pd.read_csv(Path('data/attributes/polders_data_aangevuld.csv'))
df_attributes.index = df_attributes['SHAPE_ID']

def to_m3(series, key):
    oppervlakte = df_attributes['oppervlak'].loc[key]
    series = series * oppervlakte
    series[series < 0] = 0
    return series

def get_NSE(results_xr_obj, NSE_key='NSE_1D'):
    # NSE_key = 'NSE_1D'
    if NSE_key in results_xr_obj.keys():
        return round(results_xr_obj[NSE_key],2)
    else:
        return np.nan


In [59]:
results_list[0][1]['AFVG23']['1h']

{'xr': <xarray.Dataset> Size: 146kB
 Dimensions:     (date: 731, time_step: 24)
 Coordinates:
   * date        (date) datetime64[ns] 6kB 2021-01-01 2021-01-02 ... 2023-01-01
   * time_step   (time_step) int64 192B 0 1 2 3 4 5 6 7 ... 17 18 19 20 21 22 23
 Data variables:
     afvoer_obs  (date, time_step) float32 70kB 3.376e-08 3.376e-08 ... nan nan
     afvoer_sim  (date, time_step) float32 70kB 2.813e-08 2.732e-08 ... nan nan,
 'NSE_1h': 0.14978688955307007}

In [41]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import threading, time, webbrowser

# Assumed pre-defined functions and variables:
# results_list, get_NSE, to_m3, etc.
# Structure:
#   For each model i in 0..N-1:
#       results_list[i][0] --> validation results for model i
#       results_list[i][1] --> testing results for model i
#   Each of these dictionaries is keyed by series (e.g. "area1", "area2", ...)

# Use the first model’s keys to get the list of series
series_keys = list(results_list[0][0].keys())

def create_figure(series_key):
    """
    Build the complete figure (with observed and simulation traces)
    for the given series key.
    Returns a tuple (fig, simulation_trace_map) where simulation_trace_map is a
    dict mapping model keys (e.g. "model1") to a list of trace indices corresponding
    to that model's simulation traces.
    """
    fig = go.Figure()
    simulation_trace_map = {}
    index_counter = 0  # global trace index counter

    # ---------------------------
    # Observed Data – Daily (always visible)
    # ---------------------------
    x_obs_daily_val = results_list[0][0][series_key]['1D']['xr']['date'].values
    y_obs_daily_val = to_m3(results_list[0][0][series_key]['1D']['xr']['afvoer_obs'].values.reshape(-1), series_key)
    fig.add_trace(go.Scatter(
        x=x_obs_daily_val,
        y=y_obs_daily_val,
        mode='lines',
        name='Geobserveerde validatie data (dagelijks)',
        line=dict(color='grey', dash='dash')
    ))
    index_counter += 1

    x_obs_daily_test = results_list[0][1][series_key]['1D']['xr']['date'].values
    y_obs_daily_test = to_m3(results_list[0][1][series_key]['1D']['xr']['afvoer_obs'].values.reshape(-1), series_key)
    fig.add_trace(go.Scatter(
        x=x_obs_daily_test,
        y=y_obs_daily_test,
        mode='lines',
        name='Geobserveerde testing data (dagelijks)',
        line=dict(color='grey', dash='dash')
    ))
    index_counter += 1

    # ---------------------------
    # Simulation Traces – Daily (one pair per model)
    # ---------------------------
    num_models = len(results_list)
    color_seq = px.colors.qualitative.Plotly

    for i in range(num_models):
        model_key = f"model{i+1}"
        simulation_trace_map[model_key] = []
        color = color_seq[i % len(color_seq)]

        # Daily validation simulation for model i
        x_sim_daily_val = results_list[i][0][series_key]['1D']['xr']['date'].values
        y_sim_daily_val = to_m3(results_list[i][0][series_key]['1D']['xr']['afvoer_sim'].values.reshape(-1), series_key)
        nse_daily_val = get_NSE(results_list[i][0][series_key]['1D'], NSE_key='NSE_1D')
        name_sim_daily_val = f'Gesimuleerde validatie data (dagelijks), model {i+1}, NSE: {nse_daily_val}'
        fig.add_trace(go.Scatter(
            x=x_sim_daily_val,
            y=y_sim_daily_val,
            mode='lines',
            name=name_sim_daily_val,
            line=dict(color=color, dash='dash')
        ))
        simulation_trace_map[model_key].append(index_counter)
        index_counter += 1

        # Daily testing simulation for model i
        x_sim_daily_test = results_list[i][1][series_key]['1D']['xr']['date'].values
        y_sim_daily_test = to_m3(results_list[i][1][series_key]['1D']['xr']['afvoer_sim'].values.reshape(-1), series_key)
        nse_daily_test = get_NSE(results_list[i][1][series_key]['1D'], NSE_key='NSE_1D')
        name_sim_daily_test = f'Gesimuleerde testing data (dagelijks), model {i+1}, NSE: {nse_daily_test}'
        fig.add_trace(go.Scatter(
            x=x_sim_daily_test,
            y=y_sim_daily_test,
            mode='lines',
            name=name_sim_daily_test,
            line=dict(color=color, dash='dash')
        ))
        simulation_trace_map[model_key].append(index_counter)
        index_counter += 1

    # ---------------------------
    # Observed Data – Hourly (always visible)
    # ---------------------------
    x_obs_hourly_val = (results_list[0][0][series_key]['1h']['xr']['date'].values.reshape(-1, 1)
                        + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten()
    y_obs_hourly_val = to_m3(results_list[0][0][series_key]['1h']['xr']['afvoer_obs'].values.reshape(-1), series_key)
    fig.add_trace(go.Scatter(
        x=x_obs_hourly_val,
        y=y_obs_hourly_val,
        mode='lines',
        name='Geobserveerde validatie data (uurlijks)',
        line=dict(color='grey')
    ))
    index_counter += 1

    x_obs_hourly_test = (results_list[0][1][series_key]['1h']['xr']['date'].values.reshape(-1, 1)
                         + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten()
    y_obs_hourly_test = to_m3(results_list[0][1][series_key]['1h']['xr']['afvoer_obs'].values.reshape(-1), series_key)
    fig.add_trace(go.Scatter(
        x=x_obs_hourly_test,
        y=y_obs_hourly_test,
        mode='lines',
        name='Geobserveerde testing data (uurlijks)',
        line=dict(color='grey')
    ))
    index_counter += 1

    # ---------------------------
    # Simulation Traces – Hourly (one pair per model)
    # ---------------------------
    for i in range(num_models):
        model_key = f"model{i+1}"
        color = color_seq[i % len(color_seq)]

        # Hourly validation simulation for model i
        x_sim_hourly_val = (results_list[i][0][series_key]['1h']['xr']['date'].values.reshape(-1, 1)
                            + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten()
        y_sim_hourly_val = to_m3(results_list[i][0][series_key]['1h']['xr']['afvoer_sim'].values.reshape(-1), series_key)
        nse_hourly_val = get_NSE(results_list[i][0][series_key]['1h'], NSE_key='NSE_1h')
        name_sim_hourly_val = f'Gesimuleerde validatie data (uurlijks), model {i+1}, NSE: {nse_hourly_val}'
        fig.add_trace(go.Scatter(
            x=x_sim_hourly_val,
            y=y_sim_hourly_val,
            mode='lines',
            name=name_sim_hourly_val,
            line=dict(color=color)
        ))
        simulation_trace_map[model_key].append(index_counter)
        index_counter += 1

        # Hourly testing simulation for model i
        x_sim_hourly_test = (results_list[i][1][series_key]['1h']['xr']['date'].values.reshape(-1, 1)
                             + np.arange(0, 24, 1) * np.timedelta64(1, 'h')).flatten()
        y_sim_hourly_test = to_m3(results_list[i][1][series_key]['1h']['xr']['afvoer_sim'].values.reshape(-1), series_key)
        nse_hourly_test = get_NSE(results_list[i][1][series_key]['1h'], NSE_key='NSE_1h')
        name_sim_hourly_test = f'Gesimuleerde testing data (uurlijks), model {i+1}, NSE: {nse_hourly_test}'
        fig.add_trace(go.Scatter(
            x=x_sim_hourly_test,
            y=y_sim_hourly_test,
            mode='lines',
            name=name_sim_hourly_test,
            line=dict(color=color)
        ))
        simulation_trace_map[model_key].append(index_counter)
        index_counter += 1

    # Layout update
    fig.update_layout(
        title=f"Time Series for {series_key}",
        xaxis_title="Datum",
        yaxis_title="Debiet (m3/s)",
        uirevision='constant',
        legend=dict(
            orientation='h',
            y=-0.2,
            x=0.5,
            yanchor='top',
            xanchor='center'
        ),
        margin=dict(b=100)
    )
    return fig, simulation_trace_map

# ---------------------------
# Dash App Setup
# ---------------------------
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H2("Time Series Plot with Series Dropdown and Toggles"),

    # Series selector
    html.Div(
        dcc.Dropdown(
            id='series-dropdown',
            options=[{'label': key, 'value': key} for key in series_keys],
            value=series_keys[0],
            clearable=False,
            style={'width': '300px'}
        ), style={'margin-bottom': '20px'}
    ),

    # All toggles inline: Model, Frequency, Subset
    html.Div([
        dcc.Checklist(
            id='model-checklist',
            options=[{'label': f'Model {i+1}', 'value': f'model{i+1}'} for i in range(len(results_list))],
            value=[f'model{i+1}' for i in range(len(results_list))],
            labelStyle={'display': 'inline-block'}
        ),
        dcc.Checklist(
            id='frequency-checklist',
            options=[
                {'label': 'Dagelijks', 'value': 'daily'},
                {'label': 'Uurlijks', 'value': 'hourly'}
            ],
            value=['daily', 'hourly'],
            labelStyle={'display': 'inline-block'}
        ),
        dcc.Checklist(
            id='subset-checklist',
            options=[
                {'label': 'Validatie', 'value': 'validation'},
                {'label': 'Testing', 'value': 'testing'}
            ],
            value=['validation', 'testing'],
            labelStyle={'display': 'inline-block'}
        )
    ], style={'display': 'flex', 'gap': '30px', 'alignItems': 'center', 'margin-bottom': '20px'}),

    # Graph
    dcc.Graph(id='time-series-graph')
])

@app.callback(
    Output('time-series-graph', 'figure'),
    Input('model-checklist', 'value'),
    Input('frequency-checklist', 'value'),
    Input('subset-checklist', 'value'),
    Input('series-dropdown', 'value')
)
def update_figure(selected_models, selected_freq, selected_subset, selected_series):
    fig, simulation_trace_map = create_figure(selected_series)

    # initial model-based visibility for simulation traces
    for model_key, trace_indices in simulation_trace_map.items():
        show_model = (model_key in selected_models)
        for idx in trace_indices:
            fig.data[idx].visible = show_model

    # map trace index to model for simulation
    index2model = {idx: mk for mk, lst in simulation_trace_map.items() for idx in lst}

    # apply frequency & subset toggles across all traces
    for i, trace in enumerate(fig.data):
        name_lower = trace.name.lower()
        # frequency flags
        is_daily  = 'dagelijks' in name_lower
        is_hourly = 'uurlijks' in name_lower
        ok_freq = (is_daily and 'daily' in selected_freq) or (is_hourly and 'hourly' in selected_freq)

        # subset flags
        is_val  = 'validatie' in name_lower
        is_test = 'testing' in name_lower
        ok_subset = (is_val and 'validation' in selected_subset) or (is_test and 'testing' in selected_subset)

        if i in index2model:
            # simulation: respect model, frequency & subset
            trace.visible = ok_freq and ok_subset and (index2model[i] in selected_models)
        else:
            # observed: respect only frequency & subset
            trace.visible = ok_freq and ok_subset

    return fig

# --- launch Dash from a notebook & open your browser (pure Dash) ---
def _run():
    # In notebooks, disable debug/reloader so it doesn’t block/double-run
    app.run_server(host="127.0.0.1", port=8050, debug=False, use_reloader=False)

threading.Thread(target=_run, daemon=True).start()
time.sleep(1.0)  # give the server a moment to start

# Try to open your default browser; if popups are blocked, use the printed URL
try:
    webbrowser.open("http://127.0.0.1:8050/")
finally:
    print("Dash is running at http://127.0.0.1:8050/")


Dash is running at http://127.0.0.1:8050/


In [60]:

# C:\Users\christiaan.wewer\Documents\projecten\rijnland_lstm_model_2025_project_folder\visualisatie_runs_final_runs.ipynb
d = df_attributes.copy()

In [61]:
# d['geom_simple] is the geometry column. its now a string but we should turn it into the geometry column



d.geom_simple[0]

'POLYGON ((131970.05516990824 447880.48400438216, 132107.18736638024 447846.47483294405, 132230.13556430215 447712.400700945, 132483.4691912769 447566.61343097105, 133141.5543022998 447352.293522819, 133283.87884350852 446928.8869558051, 133221.26300239726 446740.0193490258, 133324.00888461393 446683.9423286342, 133598.74094357618 446705.0081041185, 133778.1058433048 446904.508279584, 133813.61515758734 447182.5051377246, 133759.30919798315 447327.3729673376, 133636.35814788594 447472.5349601542, 133752.72850605068 447394.1564932074, 133936.62555416016 447048.4732996903, 133722.61157405603 446771.23373882554, 133598.59777049214 446671.63099236134, 133461.18309201792 446649.9705136218, 133603.50766077032 446215.4478309819, 133699.40202737012 446159.40805256297, 133760.3967217854 445970.00779785664, 132942.10512036228 445717.6620417609, 132782.92026041308 445440.2206318773, 132830.35118883435 445295.3730247087, 132726.68833078546 445151.19789960555, 132629.3392367384 444884.61177164764, 

In [62]:
import geopandas as gpd
from shapely import wkt

# If you already have d (a pandas DataFrame):

geom = d['geom_simple'].apply(lambda s: wkt.loads(s) if isinstance(s, str) and s.strip() else None)

gpd_area = gpd.GeoDataFrame(d, geometry=geom, crs="EPSG:28992")  



# (optional) drop the original text column

# sanity check
type(gpd_area.geometry.iloc[0]) 

shapely.geometry.polygon.Polygon

In [ ]:
# save areas as .geojson
# gpd_area.to_file('polders.geojson', driver='GeoJSON')

In [63]:
# warp the geometries to EPSG:28992
gpd_area = gpd_area.to_crs(epsg=28992)
# save
# gpd_area.to_file('polders_28992.geojson', driver='GeoJSON')

In [64]:
gpd_area

,SHAPE_ID,VALUE,geom_simple,landgebruik,bodem,maaiveldhoogte,maaiveldhoogte_median,maaiveldhoogte_p95_minus_p05,maaiveldhoogte_iqr,water_percentage,...,veen,zand,stuw,peil_range,kwel_mean,peil_min,peil_max,peil_mean,peil_std,geometry
SHAPE_ID,,,,,,,,,,,,,,,,,,,,,
AFVG1,AFVG1,0.064,POLYGON ((131970.05516990824 447880.4840043821...,Weidehooi,Klei op veen,-0.51,-0.84,2.90,1.12,7.8,...,0,0,0,4.180,0.128947,-3.190,0.990,-1.459201,0.971109,"POLYGON ((131970.055 447880.484, 132107.187 44..."
AFVG5,AFVG5,0.063,POLYGON ((113139.55252001957 446133.7861170391...,Weidehooi,Kleidek op veen,-1.64,-1.71,1.36,0.24,12.8,...,0,0,0,4.300,0.059957,-3.850,0.450,-2.612440,0.675826,"POLYGON ((113139.553 446133.786, 113154.033 44..."
AFVG6,AFVG6,0.067,"POLYGON ((134621.23799433495 453287.287127394,...",Weidehooi,Klei met zware tussenlaag of ondergrond,0.76,0.46,3.32,1.20,7.5,...,0,0,0,2.400,0.019342,-1.950,0.450,-0.705833,0.625378,"POLYGON ((134621.238 453287.287, 135431.606 45..."
AFVG8,AFVG8,0.068,"POLYGON ((133689.48040573893 456673.501488156,...",Bedrijventerrein,Klei met zware tussenlaag of ondergrond,1.77,1.55,3.80,0.37,3.1,...,0,0,0,1.125,-0.412742,-1.425,-0.300,-0.918750,0.555793,"POLYGON ((133689.48 456673.501, 133155.354 456..."
AFVG13,AFVG13,0.084,"POLYGON ((118640.0312079672 463013.7405577856,...",Weidehooi,Veraarde bovengrond op diep veen,-2.16,-2.16,0.94,0.41,12.5,...,1,0,0,1.765,-0.324333,-4.615,-2.850,-3.770952,0.385839,"POLYGON ((118640.031 463013.741, 118736.002 46..."
AFVG15,AFVG15,0.032,POLYGON ((126808.06131001688 457586.6995034386...,Woongebied,Licht klei met homogeen profiel,0.08,0.02,1.37,0.56,10.6,...,0,0,0,0.450,-0.753559,-1.050,-0.600,-0.866667,0.236291,"POLYGON ((126808.061 457586.7, 126786.814 4574..."
AFVG16,AFVG16,0.036,POLYGON ((114478.19845498307 455357.5741736719...,Weidehooi,Veraarde bovengrond op diep veen,-1.71,-1.94,1.85,0.60,13.5,...,1,0,0,2.945,-0.021863,-3.770,-0.825,-2.324516,0.919632,"POLYGON ((114478.198 455357.574, 114377.611 45..."
AFVG18,AFVG18,0.048,MULTIPOLYGON (((121664.3805398847 451677.10942...,Weidehooi,Klei met zware tussenlaag of ondergrond,-0.76,-0.87,2.10,1.08,6.1,...,0,0,0,2.620,-0.245467,-3.220,-0.600,-1.640909,0.629125,"MULTIPOLYGON (((121664.381 451677.109, 121636...."
AFVG22,AFVG22,0.040,POLYGON ((121104.76960907866 455118.8852393594...,Overige wegdelen,Klei met zware tussenlaag of ondergrond,-0.15,-0.22,2.45,0.92,4.3,...,0,0,0,0.825,-0.419292,-2.535,-1.710,-2.122500,0.583363,"POLYGON ((121104.77 455118.885, 121517.421 454..."


In [65]:
gpd_area.index = gpd_area['SHAPE_ID']
gpd_area = gpd_area[['SHAPE_ID', 'geom_simple']]
gpd_area.head()

,SHAPE_ID,geom_simple
SHAPE_ID,,
AFVG1,AFVG1,POLYGON ((131970.05516990824 447880.4840043821...
AFVG5,AFVG5,POLYGON ((113139.55252001957 446133.7861170391...
AFVG6,AFVG6,"POLYGON ((134621.23799433495 453287.287127394,..."
AFVG8,AFVG8,"POLYGON ((133689.48040573893 456673.501488156,..."
AFVG13,AFVG13,"POLYGON ((118640.0312079672 463013.7405577856,..."


In [66]:
# we make a table with NSE values of each model
# we have one table for validation and one for testing
# gpd_area = gpd.read_file()
# # gpd_area = gpd_area.set_index('SHAPE_ID')
# gpd_area.index = gpd_area['SHAPE_ID']
# gpd_area.drop(columns=['VALUE'], inplace=True)
gpd_model_results = {}

for i, model_result in enumerate(results_list):
    model_name = f"model_{i+1}"
    validation_results = model_result[0]
    testing_results = model_result[1]

    # Validation results
    nse_values_val_1D = {key: get_NSE(value['1D'], NSE_key='NSE_1D') for key, value in validation_results.items()}
    nse_values_val_1H = {key: get_NSE(value['1h'], NSE_key='NSE_1h') for key, value in validation_results.items()}
    df_val = pd.DataFrame({'NSE_1D':nse_values_val_1D, 'NSE_1h':nse_values_val_1H})
    df_val['NSE_AVG_1D_1h'] = (df_val['NSE_1D'] + df_val['NSE_1h']) / 2
    df_val['NSE_1D'] = df_val['NSE_1D'].round(2)
    df_val['NSE_1h'] = df_val['NSE_1h'].round(2)
    df_val['NSE_AVG_1D_1h'] = df_val['NSE_AVG_1D_1h'].round(2)
    df_val = gpd_area.join(df_val, how='inner')
    
    df_val['geom_simple']=df_val['geom_simple'].apply(lambda s: wkt.loads(s) if isinstance(s, str) and s.strip() else None)

    # Testing results
    nse_values_test_1D = {key: get_NSE(value['1D'], NSE_key='NSE_1D') for key, value in testing_results.items()}
    nse_values_test_1H = {key: get_NSE(value['1h'], NSE_key='NSE_1h') for key, value in testing_results.items()}
    df_test = pd.DataFrame({'NSE_1D':nse_values_test_1D, 'NSE_1h':nse_values_test_1H})
    df_test['NSE_AVG_1D_1h'] = (df_test['NSE_1D'] + df_test['NSE_1h']) / 2
    df_test['NSE_1D'] = df_test['NSE_1D'].round(2)
    df_test['NSE_1h'] = df_test['NSE_1h'].round(2)
    df_test['NSE_AVG_1D_1h'] = df_test['NSE_AVG_1D_1h'].round(2)
    df_test = gpd_area.join(df_test, how='inner')

    # gpd_model_results[model_name] = {
    #     'validation_gdf': df_val,
    #     'testing_gdf': df_test,
    # }
    df_test['geom_simple'] = df_test['geom_simple'].apply(lambda s: wkt.loads(s) if isinstance(s, str) and s.strip() else None)

    gpd_model_results[model_name] = {
        'validation_gdf': gpd.GeoDataFrame(df_val, geometry='geom_simple', crs='EPSG:28992'),
        'testing_gdf': gpd.GeoDataFrame(df_test, geometry='geom_simple', crs='EPSG:28992'),
    }

In [ ]:
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import geopandas as gpd
import json

# ------------------------------------------------------------------------------
# Assume gpd_model_results is a dictionary structured as:
# {
#     "model_1": {"validation_gdf": <GeoDataFrame>, "testing_gdf": <GeoDataFrame>},
#     "model_2": {"validation_gdf": <GeoDataFrame>, "testing_gdf": <GeoDataFrame>},
#     ...
# }
# Each GeoDataFrame should include a column for a unique area identifier (e.g., "SHAPE_ID").
# ------------------------------------------------------------------------------

# Ensure each GeoDataFrame's index is set to "SHAPE_ID" if that’s the identifier.
for model in gpd_model_results:
    for dataset in ['validation_gdf', 'testing_gdf']:
        gdf = gpd_model_results[model][dataset]
        # if "SHAPE_ID" in gdf.columns:
        #     gpd_model_results[model][dataset] = gdf.set_index("SHAPE_ID")

# List of available models (keys of gpd_model_results)
model_names = list(gpd_model_results.keys())

# List of available NSE metrics to display
nse_options = ["NSE_1D", "NSE_1h", "NSE_AVG_1D_1h"]

# ------------------------------------------------------------------------------
# Create Dash app layout with three dropdowns and one graph component.
# ------------------------------------------------------------------------------
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("NSE Map Dashboard"),
    
    # Dropdown to select a model
    html.Div([
        html.Label("Select Model:"),
        dcc.Dropdown(
            id="model-dropdown",
            options=[{"label": model, "value": model} for model in model_names],
            value=model_names[0]
        )
    ], style={'width': '30%', 'display': 'inline-block', 'padding': '10px'}),
    
    # Dropdown to select the dataset (validation or testing)
    html.Div([
        html.Label("Select Dataset:"),
        dcc.Dropdown(
            id="dataset-dropdown",
            options=[
                {"label": "Validation", "value": "validation_gdf"},
                {"label": "Testing", "value": "testing_gdf"}
            ],
            value="validation_gdf"
        )
    ], style={'width': '30%', 'display': 'inline-block', 'padding': '10px'}),
    
    # Dropdown to select the NSE metric to display
    html.Div([
        html.Label("Select NSE Metric:"),
        dcc.Dropdown(
            id="nse-dropdown",
            options=[{"label": metric, "value": metric} for metric in nse_options],
            value="NSE_AVG_1D_1h"
        )
    ], style={'width': '30%', 'display': 'inline-block', 'padding': '10px'}),
    
    # Graph component to display the map
    dcc.Graph(id="map-graph")
])

# ------------------------------------------------------------------------------
# Callback: Updates the map based on selected model, dataset, and NSE metric.
# ------------------------------------------------------------------------------
@app.callback(
    Output("map-graph", "figure"),
    Input("model-dropdown", "value"),
    Input("dataset-dropdown", "value"),
    Input("nse-dropdown", "value")
)
def update_map(model, dataset_key, nse_metric):
    # Retrieve a copy of the selected GeoDataFrame
    gdf = gpd_model_results[model][dataset_key].copy()
    
    # Debug: Check if GeoDataFrame is empty
    if gdf.empty:
        print("Warning: The GeoDataFrame is empty!")
        return {}
    
    for col in ["NSE_1D", "NSE_1h", "NSE_AVG_1D_1h"]:
        if col in gdf.columns:
            gdf[col] = pd.to_numeric(gdf[col], errors="coerce")
    
    # --------------------------------------------------------------------------
    # Step 1: Reproject to EPSG:4326 (WGS84 lat/lon), required for Plotly maps.
    # --------------------------------------------------------------------------
    centroids = gdf.geometry.centroid.to_crs("EPSG:4326")
    ys = centroids.y
    xs = centroids.x
    gdf = gdf.to_crs("EPSG:4326")
    
    # --------------------------------------------------------------------------
    # Step 2: Convert the GeoDataFrame to GeoJSON format.
    # --------------------------------------------------------------------------
    geojson_str = gdf.to_json()
    geojson_data = json.loads(geojson_str)
    
    # Debug: Print GeoJSON info (first feature) to verify proper conversion
    if geojson_data.get("features"):
        print("First GeoJSON feature properties:", geojson_data["features"][0]["properties"])
    else:
        print("Warning: No features found in GeoJSON data!")
    center = {
        "lat": ys.mean(),
        "lon": xs.mean()
    }
    
    # --------------------------------------------------------------------------
    # Step 3: Create the choropleth mapbox figure.
    # --------------------------------------------------------------------------
    fig = px.choropleth_map(
        gdf,
        geojson=geojson_data,
        locations="SHAPE_ID",
        featureidkey="properties.SHAPE_ID",
        color=nse_metric,
        zoom=9.5,
        center=center,
        opacity=0.5,
        color_continuous_scale="RdYlGn",
        labels={nse_metric: nse_metric}, 
        range_color=[-1, 1],
        # labels="SHAPE_ID",
        # labels=gdf.index
        # fitbounds="locations",
    )
    

    # Ensure choropleth is drawn below other layers so text isn't hidden:
    # if fig.data:
    fig.data[0].update(below=True)

    # # 2. Compute centroid coordinates for each polygon (longitude and latitude)
    # gdf["centroid_lon"] = gdf.geometry.centroid.x
    # gdf["centroid_lat"] = gdf.geometry.centroid.y

    # 3. Add a Scattermapbox trace with text labels at those centroids
    fig.add_trace(go.Scattermap(
        lon=xs, lat=ys,
        text=(gdf['POLDERNAAM'].astype(str) if 'POLDERNAAM' in gdf.columns else gdf['SHAPE_ID'].astype(str)),
        mode="text",
        textfont=dict(color="black", size=10),
        showlegend=False,
        hoverinfo="skip"  # skip hover on labels to avoid duplicating info
    ))

 
    # Set the map style and adjust margins
    fig.update_layout(mapbox_style="carto-positron",   
        margin={"r":0, "t":0, "l":0, "b":0},
        # annotations=annotations,
    )


    return fig

# if __name__ == "__main__":
#     app.run_server(debug=True, port=8051)  # Use a different port to avoid conflicts with the previous app

# --- launch Dash from a notebook & open your browser (pure Dash) ---
def _run():
    # In notebooks, disable debug/reloader so it doesn’t block/double-run
    app.run_server(host="127.0.0.1", port=8051, debug=False, use_reloader=False)

threading.Thread(target=_run, daemon=True).start()
time.sleep(1.0)  # give the server a moment to start

# Try to open your default browser; if popups are blocked, use the printed URL
try:
    webbrowser.open("http://127.0.0.1:8051/")
finally:
    print("Dash is running at http://127.0.0.1:8051/")


First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': 0.72, 'NSE_1h': 0.66, 'NSE_AVG_1D_1h': 0.69}
Dash is running at http://127.0.0.1:8051/


First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': 0.72, 'NSE_1h': 0.66, 'NSE_AVG_1D_1h': 0.69}
First GeoJSON feature properties: {'SHAPE_ID': 'AFVG1', 'NSE_1D': 0.75, 'NSE_1h': 0.68, 'NSE_AVG_1D_1h': 0.72}
